# SQLite + LangGraph: Text-to-SQL 실습

**목표:** Python SQLite → SQL 기본 조회 → Chinook DB 스키마 → 질문을 SQL로 변환 → LangGraph 실행

- `hello.db`: 직접 학생 데이터 생성/조회
- `Chinook.db`: 실제 관계형 예제 데이터로 JOIN/집계 연습
- OpenAI API 키가 없으면 준비된 질문 두 개로 **SQL 검색 흐름**을 검증
- 키가 있으면 질문 → SQL 생성 → 안전하게 조회 → 답변하는 기본 흐름 실행

> PDF RAG는 문서를 검색하고, Text-to-SQL은 DB **스키마를 참고해 조회 SQL**을 만든다는 차이가 있어요.

## 1. SQLite 연결과 테이블 생성

`sqlite3`는 파이썬 내장 모듈. `?` 바인딩으로 값을 안전하게 삽입한다.

In [1]:
import sqlite3
from pathlib import Path

DATA_DIR = Path.cwd()
conn_simple = sqlite3.connect(DATA_DIR / "hello.db")

conn_simple.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    score INTEGER NOT NULL
)
""")

students = [(1, "김사과", 90), (2, "반하나", 85), (3, "오렌지", 95)]
conn_simple.executemany(
    "INSERT OR REPLACE INTO students (id, name, score) VALUES (?, ?, ?)",
    students
)
conn_simple.commit()
print("학생 데이터 저장 완료")

학생 데이터 저장 완료


In [2]:
# WHERE, ORDER BY와 ? 바인딩
min_score = 90
rows = conn_simple.execute(
    "SELECT id, name, score FROM students WHERE score >= ? ORDER BY score DESC",
    (min_score,)
).fetchall()
for row in rows:
    print(row)

(3, '오렌지', 95)
(1, '김사과', 90)


## 2. Chinook DB 읽기

Chinook은 고객(Customer), 주문(Invoice), 트랙(Track) 등으로 구성된 샘플 데이터베이스다. 파일이 없으면 제공 URL에서 다운로드한다.

In [3]:
from urllib.request import urlopen

path = DATA_DIR / "Chinook.db"
if not path.exists():
    url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
    with urlopen(url, timeout=30) as response:
        path.write_bytes(response.read())

conn = sqlite3.connect(path)
conn.row_factory = sqlite3.Row

tables = conn.execute("""
    SELECT name FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""").fetchall()
print("DB 경로:", path.resolve())
print("테이블:", [r["name"] for r in tables])

DB 경로: C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\01_class\2026_10_30_agent\Chinook.db
테이블: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [4]:
# Customer, Invoice 컬럼 구조 확인
for table in ("Customer", "Invoice"):
    print(f"\n[{table}]")
    for col in conn.execute(f'PRAGMA table_info("{table}")'):
        print(col["name"], col["type"])


[Customer]
CustomerId INTEGER
FirstName NVARCHAR(40)
LastName NVARCHAR(20)
Company NVARCHAR(80)
Address NVARCHAR(70)
City NVARCHAR(40)
State NVARCHAR(40)
Country NVARCHAR(40)
PostalCode NVARCHAR(10)
Phone NVARCHAR(24)
Fax NVARCHAR(24)
Email NVARCHAR(60)
SupportRepId INTEGER

[Invoice]
InvoiceId INTEGER
CustomerId INTEGER
InvoiceDate DATETIME
BillingAddress NVARCHAR(70)
BillingCity NVARCHAR(40)
BillingState NVARCHAR(40)
BillingCountry NVARCHAR(40)
BillingPostalCode NVARCHAR(10)
Total NUMERIC(10,2)


In [ ]:
# JOIN + GROUP BY: 지출이 많은 고객 5명
sql = """
SELECT c.FirstName || ' ' || c.LastName AS customer_name,
       COUNT(i.InvoiceId) AS invoice_count,
       ROUND(SUM(i.Total), 2) AS total_spent
FROM Customer AS c
JOIN Invoice AS i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId
ORDER BY total_spent DESC
LIMIT 5
"""
for row in conn.execute(sql):
    print(dict(row))

{'customer_name': 'Helena Holý', 'invoice_count': 7, 'total_spent': 49.62}
{'customer_name': 'Richard Cunningham', 'invoice_count': 7, 'total_spent': 47.62}
{'customer_name': 'Luis Rojas', 'invoice_count': 7, 'total_spent': 46.62}
{'customer_name': 'Ladislav Kovács', 'invoice_count': 7, 'total_spent': 45.62}
{'customer_name': "Hugh O'Reilly", 'invoice_count': 7, 'total_spent': 45.62}


## 3. 모델에 DB 스키마 제공하기

모델은 `Chinook.db` 파일 구조를 자동으로 알지 못하므로, SQL 생성 전에 **CREATE TABLE 문**을 읽어 전달한다. 실습 범위는 Customer·Invoice 2개 테이블로 제한한다.

In [6]:
allowed_tables = ("Customer", "Invoice")
schema = []

for table in allowed_tables:
    row = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type = 'table' AND name = ?",
        (table,)
    ).fetchone()
    schema.append(row["sql"])

schema_text = "\n\n".join(schema)
print(schema_text)

CREATE TABLE [Customer]
(
    [CustomerId] INTEGER  NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    CONSTRAINT [PK_Customer] PRIMARY KEY  ([CustomerId]),
    FOREIGN KEY ([SupportRepId]) REFERENCES [Employee] ([EmployeeId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
)

CREATE TABLE [Invoice]
(
    [InvoiceId] INTEGER  NOT NULL,
    [CustomerId] INTEGER  NOT NULL,
    [InvoiceDate] DATETIME  NOT NULL,
    [BillingAddress] NVARCHAR(70),
    [BillingCity] NVARCHAR(40),
    [BillingState] NVARCHAR(40),
    [BillingCountry] NVARCHAR(40),
    [BillingPostalCode] NVARCHAR(10),
    [Total] NUMERIC(10,2)  NOT NULL,
    CONSTRAINT [PK_Invoice] PRIMARY KEY  ([InvoiceId]),
    F

## 4. LangGraph 노드 구현

`write_sql`은 SQL 생성, `run_sql`은 실행, `format_answer`는 결과를 표시한다.

미설치 시: `%pip install langgraph langchain-openai python-dotenv`

실습용 가드: 조회문(SELECT)만 허용하고 DB 자체를 **읽기 전용**으로 연다. 실제 서비스는 SQL AST 검사, 테이블별 접근 제어, 실행 시간 제한 등이 더 필요하다.

In [7]:
import os
import re
from contextlib import closing
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

llm = None
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"), temperature=0)

# 키가 없을 때도 전체 노드 흐름을 확인할 수 있는 예제 SQL
sample_sql = {
    "고객별 주문금액 상위 5명": """
        SELECT c.FirstName || ' ' || c.LastName AS name,
               ROUND(SUM(i.Total), 2) AS total_spent
        FROM Customer AS c
        JOIN Invoice AS i ON c.CustomerId = i.CustomerId
        GROUP BY c.CustomerId
        ORDER BY total_spent DESC
        LIMIT 5
    """,
    "국가별 고객 수 상위 5개": """
        SELECT Country, COUNT(*) AS customer_count
        FROM Customer
        GROUP BY Country
        ORDER BY customer_count DESC
        LIMIT 5
    """
}

class SQLState(TypedDict, total=False):
    question: str
    sql: str
    rows: list[dict]
    error: str
    answer: str

def write_sql(state: SQLState):
    question = state["question"]
    if llm is None:
        sql = sample_sql.get(question)
        if not sql:
            return {"sql": "", "error": "API 키가 없어 예제 질문만 지원합니다."}
        return {"sql": sql.strip()}

    prompt = f"""SQLite SELECT문 하나만 작성하세요. 설명과 코드블록 없이 SQL만 답하세요.
허용 테이블은 Customer, Invoice뿐입니다. 결과는 LIMIT 10으로 제한하세요.
스키마:
{schema_text}
질문: {question}
"""
    result = llm.invoke(prompt)
    sql = str(result.content).strip()
    sql = re.sub(r"^\s*```(?:sql)?\s*|\s*```\s*$", "", sql, flags=re.I).strip()
    return {"sql": sql}

def run_sql(state: SQLState):
    if state.get("error"):
        return {"rows": []}
    sql = state.get("sql", "").strip()
    if not re.match(r"^SELECT\b", sql, re.I):
        return {"rows": [], "error": "조회용 SELECT 문만 허용합니다."}
    try:
        # DB 자체를 읽기 전용으로 열고 최대 10행만 반환
        uri = path.resolve().as_uri() + "?mode=ro"
        with closing(sqlite3.connect(uri, uri=True)) as readonly:
            readonly.row_factory = sqlite3.Row
            readonly.execute("PRAGMA query_only=ON")
            rows = readonly.execute(sql).fetchmany(10)
            return {"rows": [dict(row) for row in rows]}
    except sqlite3.Error as exc:
        return {"rows": [], "error": f"SQL 오류: {exc}"}

def format_answer(state: SQLState):
    if state.get("error"):
        return {"answer": state["error"]}
    if not state.get("rows"):
        return {"answer": "조회 결과가 없어 확인할 수 없습니다."}
    return {"answer": "\n".join(map(str, state["rows"]))}

In [8]:
builder = StateGraph(SQLState)
builder.add_node("write_sql", write_sql)
builder.add_node("run_sql", run_sql)
builder.add_node("format_answer", format_answer)

builder.add_edge(START, "write_sql")
builder.add_edge("write_sql", "run_sql")
builder.add_edge("run_sql", "format_answer")
builder.add_edge("format_answer", END)

app = builder.compile()
print("START → write_sql → run_sql → format_answer → END")

START → write_sql → run_sql → format_answer → END


In [ ]:
# 질문 두 개의 SQL / 결과 비교
for question in ("고객별 주문금액 상위 5명", "국가별 고객 수 상위 5개"):
    result = app.invoke({"question": question})
    print("\n질문:", question)
    print("SQL:", result.get("sql", ""))
    print("결과:", result["answer"])
    print("-" * 50)


질문: 고객별 주문금액 상위 5명
SQL: SELECT CustomerId, FirstName, LastName, TotalOrderAmount
FROM (
    SELECT c.CustomerId,
           c.FirstName,
           c.LastName,
           SUM(i.Total) AS TotalOrderAmount
    FROM Customer AS c
    JOIN Invoice AS i ON i.CustomerId = c.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName
    ORDER BY TotalOrderAmount DESC
    LIMIT 5
)
LIMIT 10;
결과: {'CustomerId': 6, 'FirstName': 'Helena', 'LastName': 'Holý', 'TotalOrderAmount': 49.62}
{'CustomerId': 26, 'FirstName': 'Richard', 'LastName': 'Cunningham', 'TotalOrderAmount': 47.62}
{'CustomerId': 57, 'FirstName': 'Luis', 'LastName': 'Rojas', 'TotalOrderAmount': 46.62}
{'CustomerId': 45, 'FirstName': 'Ladislav', 'LastName': 'Kovács', 'TotalOrderAmount': 45.62}
{'CustomerId': 46, 'FirstName': 'Hugh', 'LastName': "O'Reilly", 'TotalOrderAmount': 45.62}
--------------------------------------------------

질문: 국가별 고객 수 상위 5개
SQL: SELECT Country, COUNT(*) AS CustomerCount
FROM Customer
GROUP BY Country


: 

## 직접 확인할 것

1. `execute`와 `executemany`, `fetchall`의 차이
2. `JOIN`으로 연결되는 키(CustomerId)의 역할
3. `schema_text`가 없으면 모델이 어떤 SQL을 만들 수 있는가
4. `SQLState`에 SQL·rows·answer를 각 노드가 언제 채우는가
5. Text-to-SQL 결과가 부정확하면 **스키마 → 생성 SQL → 실행 결과** 순서로 점검하기

**다음 확장:** SQL 생성 오류가 발생했을 때 조건부 엣지로 재시도 노드를 추가해 보기.